# Qwen3-VL-32B + LoRA — same pipeline as `serve_qwen_colab.ipynb`; only **model load** uses LoRA. **Saves** use `RUN_TAG` (default `finetune`): `explanation_qwen_finetune.json`, `judge_qwen_finetune.json`, summaries — so zero-shot `*_zeroshot.json` files are untouched.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → GPU (A100 recommended for 32B)"
free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f"{free/1e9:.1f}/{total/1e9:.1f} GB free")

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=4.45.0", "accelerate>=0.30.0", "peft", "qwen-vl-utils[decord]", "openai"])

In [ ]:
from pathlib import Path

BASE_MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"
LORA_DIR = Path("/content/drive/MyDrive/qwen3vl32_rtfm_lora")

OUTPUT_DIR = Path("/content/drive/MyDrive/rtfm_pipeline_outputs")
LOCAL_FRAMES = Path("/content/qwen_frames")
ANNOTATIONS_PATH = Path("/content/drive/MyDrive/annotations.json")

MODEL_ID = BASE_MODEL_ID
RUN_TAG = "finetune"

MAX_NEW_TOKENS = 300
SLEEP_BETWEEN = 0.5

In [ ]:
import json
import shutil
from google.colab import drive

drive.mount("/content/drive")
assert (LORA_DIR / "adapter_config.json").is_file(), f"Missing adapter: {LORA_DIR}"

LOCAL_FRAMES.mkdir(parents=True, exist_ok=True)
video_metas = {}
for video_folder in sorted(OUTPUT_DIR.iterdir()):
    if not video_folder.is_dir() or video_folder.name.startswith("."):
        continue
    meta_file = video_folder / "metadata.json"
    if not meta_file.is_file():
        continue
    meta = json.loads(meta_file.read_text(encoding="utf-8"))
    vname = meta.get("video_id", video_folder.name)
    valid = []
    for fr in meta.get("extracted_frames", []):
        fname = fr.get("file")
        if not fname:
            continue
        srcp = video_folder / fname
        if not srcp.is_file():
            continue
        dst = LOCAL_FRAMES / vname / fname
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.is_file():
            shutil.copy2(srcp, dst)
        fr = dict(fr)
        fr["_abs_path"] = str(dst)
        valid.append(fr)
    if not valid:
        continue
    meta = dict(meta)
    meta["video_id"] = vname
    meta["extracted_frames"] = valid
    video_metas[vname] = meta

annotations = {}
if ANNOTATIONS_PATH.is_file():
    annotations = {e["video_id"]: e for e in json.loads(ANNOTATIONS_PATH.read_text(encoding="utf-8")) if "video_id" in e}
print(f"Loaded {len(video_metas)} videos from {OUTPUT_DIR}")
print(f"Annotations: {len(annotations)}")


In [ ]:
import gc
import os
import torch
from transformers import AutoProcessor
from peft import PeftModel

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

try:
    from transformers import Qwen3VLForConditionalGeneration as QwenVLModel
except ImportError:
    from transformers import Qwen2VLForConditionalGeneration as QwenVLModel

base = QwenVLModel.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, str(LORA_DIR))
model.eval()
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, min_pixels=256 * 28 * 28, max_pixels=512 * 28 * 28)
print("ok", torch.cuda.memory_allocated() / 1e9, "GB")

In [ ]:
import base64
import gc
import json
import os
import tempfile
import time
from pathlib import Path

import torch
from openai import OpenAI
from qwen_vl_utils import process_vision_info

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or ""
assert OPENAI_API_KEY, "Colab secret OPENAI_API_KEY"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
openai_client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """You are a surveillance video anomaly analyst. You will be shown a set of \
frames sampled from a surveillance video that has been flagged as anomalous \
by a weakly-supervised anomaly detection model (RTFM).

The frames are ordered temporally. Each frame comes from a specific temporal \
snippet of the video, and you are given the anomaly score for that snippet \
(0 = normal, 1 = highly anomalous).

The frames were specifically selected from the anomalous portions of the \
video — they represent the onset, peak, and resolution of the detected anomaly.

Your task: Based on ALL the frames and their anomaly scores together, provide \
a single concise explanation (2-3 sentences) of what anomalous activity is \
happening. Focus on:
- WHAT is happening (the specific anomalous activity)
- WHO/WHAT is involved (people, vehicles, objects — describe appearance)
- WHEN in the sequence it starts and ends
- WHY it is anomalous (how it deviates from normal pedestrian behaviour)

Respond with ONLY a JSON object in this exact format:
{\"explanation\": \"...\"}"""

JUDGE_PROMPT = """You are an impartial judge evaluating the quality of an AI-generated \
explanation of an anomalous event in a surveillance video.

You will be given:
1. A HUMAN ground-truth explanation written by someone who watched the full video.
2. An AI-GENERATED explanation produced by a vision-language model that only \
saw a few sampled frames guided by anomaly scores from a weakly-supervised \
anomaly detection model (RTFM).

Score the AI explanation on these 4 criteria (each 1-5):
- correctness  : Does the AI identify the same anomaly as the human?
- specificity  : Does the AI mention specific details (objects, people, actions, location)?
- completeness : Does the AI capture all aspects the human mentioned?
- fluency      : Is the explanation well-written, clear, and natural?

Also provide a brief justification (1-2 sentences).

Respond with ONLY a JSON object:
{\"correctness\": 1-5, \"specificity\": 1-5, \"completeness\": 1-5, \"fluency\": 1-5, \"justification\": \"...\"}"""

NORMAL_FP_GT = "There is nothing anomalous in this video. All pedestrians are walking normally."


def run_qwen_inference_local(body: dict) -> str:
    tmp_paths = []
    try:
        content = [{"type": "text", "text": body["intro_text"]}]
        for fr in body["frames"]:
            tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
            tmp.write(base64.b64decode(fr["b64"]))
            tmp.close()
            tmp_paths.append(tmp.name)
            content.append({"type": "text", "text": fr["label"]})
            content.append({"type": "image", "image": f"file://{tmp.name}"})
        messages = [
            {"role": "system", "content": body["system_prompt"]},
            {"role": "user", "content": content},
        ]
        text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)[:2]
        inputs = processor(text=[text_input], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=body.get("max_new_tokens", MAX_NEW_TOKENS), do_sample=False)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
        raw = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        text = raw
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        try:
            parsed = json.loads(text.strip())
        except json.JSONDecodeError:
            parsed = {"explanation": raw}
        return parsed.get("explanation", raw)
    except Exception as e:
        return f"ERROR: {e}"
    finally:
        for p in tmp_paths:
            try:
                os.unlink(p)
            except Exception:
                pass
        gc.collect()


def build_explain_request(meta: dict) -> dict:
    frames = meta["extracted_frames"]
    segs = meta.get("anomalous_segments", [])

    def seg_label(s):
        a = s.get("start_snippet", s.get("start", "?"))
        b = s.get("end_snippet", s.get("end", "?"))
        return f"snippets {a}-{b}"

    seg_str = ", ".join(seg_label(s) for s in segs) if segs else "unknown"
    score_str = ", ".join(f"snippet {f['snippet_idx']}={f['score']:.3f}" for f in frames)
    intro = (
        f"This video has {meta['n_segments']} temporal snippets (~16 frames each). "
        f"RTFM flagged these contiguous segments as anomalous: [{seg_str}]. "
        f"Video-level anomaly gate score: {meta['gate_score']:.3f} "
        f"(0=normal, 1=highly anomalous; threshold=0.2). "
        f"Below are {len(frames)} frames sampled from the anomalous segments. "
        f"Per-snippet anomaly scores: [{score_str}]."
    )
    frame_items = []
    for fr in frames:
        p = fr.get("_abs_path", "")
        if not Path(p).is_file():
            continue
        frame_items.append({
            "label": f"Frame from snippet {fr['snippet_idx']} (frame #{fr['frame_num']}, anomaly score: {fr['score']:.3f}):",
            "b64": base64.b64encode(Path(p).read_bytes()).decode("utf-8"),
        })
    return {"system_prompt": SYSTEM_PROMPT, "intro_text": intro, "frames": frame_items, "max_new_tokens": MAX_NEW_TOKENS}


def call_gpt4o_judge(human_expl: str, ai_expl: str) -> dict:
    user_msg = f'HUMAN ground-truth explanation:\n"{human_expl}"\n\nAI-generated explanation:\n"{ai_expl}"'
    try:
        resp = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "system", "content": JUDGE_PROMPT}, {"role": "user", "content": user_msg}],
            max_tokens=300,
            response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        return {"correctness": None, "specificity": None, "completeness": None, "fluency": None, "justification": f"ERROR: {e}"}

In [ ]:
qwen_explanations = {}
n = len(video_metas)
for i, (vname, meta) in enumerate(sorted(video_metas.items())):
    print(f"[{i+1}/{n}] {vname}", end=" ", flush=True)
    body = build_explain_request(meta)
    expl = run_qwen_inference_local(body)
    print((expl[:72] + "...") if len(expl) > 72 else expl)
    qwen_explanations[vname] = {
        "video_id": vname,
        "model": MODEL_ID,
        "gate_score": meta["gate_score"],
        "n_segments": meta["n_segments"],
        "anomalous_segments": meta.get("anomalous_segments", []),
        "n_frames_sent": len(body["frames"]),
        "explanation": expl,
    }
    vid_out = OUTPUT_DIR / vname
    vid_out.mkdir(parents=True, exist_ok=True)
    (vid_out / f"explanation_qwen_{RUN_TAG}.json").write_text(json.dumps(qwen_explanations[vname], indent=2), encoding="utf-8")

(OUTPUT_DIR / f"qwen_{RUN_TAG}_explanations_summary.json").write_text(
    json.dumps(list(qwen_explanations.values()), indent=2), encoding="utf-8")
print("saved", OUTPUT_DIR)


In [ ]:
qwen_judge_results = []
for i, (vname, expl_data) in enumerate(sorted(qwen_explanations.items())):
    ai_expl = expl_data.get("explanation", "")
    if not ai_expl or str(ai_expl).startswith("ERROR"):
        continue
    clip_id = vname.split("_")[1] if "_" in vname else vname
    is_truly_anomalous = len(clip_id) == 4
    if is_truly_anomalous and vname in annotations:
        human_expl = annotations[vname]["explanation"]
        video_type = "anomalous"
        gt_start = annotations[vname].get("anomaly_start_frame")
        gt_end = annotations[vname].get("anomaly_end_frame")
    else:
        human_expl = NORMAL_FP_GT
        video_type = "normal_FP"
        gt_start = gt_end = None
    print(f"[{i+1}] {vname} {video_type}")
    scores = call_gpt4o_judge(human_expl, ai_expl)
    row = {
        "video_id": vname,
        "model": MODEL_ID,
        "video_type": video_type,
        "human_explanation": human_expl,
        "ai_explanation": ai_expl,
        "gate_score": expl_data.get("gate_score"),
        "gt_anomaly_start": gt_start,
        "gt_anomaly_end": gt_end,
        "scores": scores,
    }
    qwen_judge_results.append(row)
    (OUTPUT_DIR / vname / f"judge_qwen_{RUN_TAG}.json").write_text(json.dumps(row, indent=2), encoding="utf-8")
    time.sleep(SLEEP_BETWEEN)

(OUTPUT_DIR / f"qwen_{RUN_TAG}_judge_summary.json").write_text(json.dumps(qwen_judge_results, indent=2), encoding="utf-8")
print(len(qwen_judge_results), "judged")
